In [1]:
import torch
from torch import nn   
net = nn.Sequential(nn.Linear(4,8), nn.ReLU(), nn.Linear(8,1))

In [2]:
X = torch.rand(size=(2,4))
net(X)

tensor([[ 0.0347],
        [-0.0528]], grad_fn=<AddmmBackward0>)

In [3]:
print(net[2].state_dict())

OrderedDict([('weight', tensor([[-0.0440,  0.2409,  0.0836, -0.2771, -0.1270, -0.2458,  0.1909, -0.2531]])), ('bias', tensor([0.1963]))])


In [4]:
print(net.state_dict())

OrderedDict([('0.weight', tensor([[-0.1552, -0.2237,  0.2926, -0.0865],
        [-0.0198,  0.0424,  0.2034, -0.4358],
        [-0.1034,  0.2014,  0.3759, -0.1504],
        [ 0.0374,  0.4477,  0.1580, -0.4903],
        [ 0.3250, -0.4185,  0.3765,  0.1679],
        [-0.1596,  0.0698,  0.2819, -0.1130],
        [-0.3430,  0.0497,  0.1885, -0.2074],
        [ 0.3108, -0.1291,  0.3642, -0.4292]])), ('0.bias', tensor([-0.2041, -0.3854, -0.2001,  0.3140,  0.3876,  0.3395, -0.3682, -0.2458])), ('2.weight', tensor([[-0.0440,  0.2409,  0.0836, -0.2771, -0.1270, -0.2458,  0.1909, -0.2531]])), ('2.bias', tensor([0.1963]))])


In [5]:
print(type(net[2].bias))
print(net[2].bias)
print(net[2].bias.data)

<class 'torch.nn.parameter.Parameter'>
Parameter containing:
tensor([0.1963], requires_grad=True)
tensor([0.1963])


In [6]:
print(net[2].bias.data.shape)

torch.Size([1])


In [7]:
net[2].weight.grad == None

True

In [9]:
print(*[(name, param.shape) for name, param in net[0].named_parameters()])
print(*[(name, param.shape) for name, param in net.named_parameters()])

('weight', torch.Size([8, 4])) ('bias', torch.Size([8]))
('0.weight', torch.Size([8, 4])) ('0.bias', torch.Size([8])) ('2.weight', torch.Size([1, 8])) ('2.bias', torch.Size([1]))


In [12]:
net.state_dict()['2.bias'].data

tensor([0.1963])

In [17]:
def block1():
    return nn.Sequential(nn.Linear(4, 8), nn.ReLU(),
                         nn.Linear(8, 4), nn.ReLU())

def block2():
    net = nn.Sequential()
    for i in range(4):
        # 在这里嵌套
        net.add_module(f'block {i}', block1())
    return net

rgnet = nn.Sequential(block2(), nn.Linear(4, 1))
rgnet(X)

tensor([[0.2178],
        [0.2178]], grad_fn=<AddmmBackward0>)

In [18]:
print(rgnet)

Sequential(
  (0): Sequential(
    (block 0): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 1): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 2): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 3): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
  )
  (1): Linear(in_features=4, out_features=1, bias=True)
)


In [19]:
rgnet[0][1][0].bias.data

tensor([-0.1041, -0.4048,  0.1541, -0.3872, -0.2406, -0.4003, -0.3612, -0.4689])

In [21]:
def init_normal(m):
    if type(m) == nn.Linear:
        nn.init.normal_(m.weight, mean = 0, std=0.01)
        nn.init.zeros_(m.bias)
net.apply(init_normal)
net[0].weight.data[0], net[0].bias.data[0]

(tensor([-0.0174,  0.0049,  0.0157, -0.0033]), tensor(0.))

In [23]:
def init_constant(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight, 1)
        nn.init.zeros_(m.bias)
net.apply(init_constant)
net[0].weight.data[0], net[0].bias.data[0]

(tensor([1., 1., 1., 1.]), tensor(0.))

In [24]:
def init_xavier(m):
    if type(m) == nn.Linear:
        nn.init.xavier_uniform_(m.weight)
def init_42(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight, 42)

net[0].apply(init_xavier)
net[2].apply(init_42)
print(net[0].weight.data[0])
print(net[2].weight.data)

tensor([ 0.1828, -0.2404,  0.0058,  0.4132])
tensor([[42., 42., 42., 42., 42., 42., 42., 42.]])


In [25]:
def my_init(m):
    if type(m) == nn.Linear:
        print('Init', *[(name, param.shape) for name, param in m.named_parameters()][0])
        nn.init.uniform_(m.weight, -10, 10)
        m.weight.data *= m.weight.data.abs() >= 5

net.apply(my_init)
net[0].weight[:2]

Init weight torch.Size([8, 4])
Init weight torch.Size([1, 8])


tensor([[ 8.1587, -7.3566,  7.8596,  0.0000],
        [ 8.4970,  0.0000,  0.0000,  9.4295]], grad_fn=<SliceBackward0>)

In [27]:
shared = nn.Linear(8, 8)
net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), shared, nn.ReLU(), shared, nn.ReLU(), nn.Linear(8, 1))
net(X)
print(net[2].weight.data[0] == net[4].weight.data[0])
net[2].weight.data[0, 0] = 100
print(net[2].weight.data[0] == net[4].weight.data[0])

tensor([True, True, True, True, True, True, True, True])
tensor([True, True, True, True, True, True, True, True])
